**Please set up your credentials JSON as GCP_CREDENTIALS secrets**

In [25]:
import os
from dotenv import load_dotenv

load_dotenv()

# Check Current Working Directory
print("📍 Current Directory:", os.getcwd())

# List all files (including hidden ones starting with .)
print("📂 Files in this folder:", os.listdir("."))

gcp_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
print(f"🔍 Looking for path: '{gcp_path}'")

if gcp_path and os.path.exists(gcp_path):
    print("✅ File found!")
else:
    print("❌ File NOT found at that path.")

📍 Current Directory: /home/andri/Git/de-zoomcamp-workshop/03-data-warehouse/homework
📂 Files in this folder: ['load-yellow-data-taxi.py', 'DLT_upload_to_GCP.ipynb', '.env', 'gcp-key.base64', 'gcp-key.json']
🔍 Looking for path: '.gcp-key.json'
❌ File NOT found at that path.


In [23]:
import os
from dotenv import load_dotenv

load_dotenv()

# Only set the path to the JSON key file
gcp_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

if not gcp_path or not os.path.exists(gcp_path):
    raise RuntimeError(
        "❌ GOOGLE_APPLICATION_CREDENTIALS is not set or file does not exist"
    )

print("✅ Using GCP credentials file:", gcp_path)

# Optional bucket default
os.environ["BUCKET_URL"] = os.getenv(
    "BUCKET_URL",
    "gs://kestra-zoomcamp-de-demo-2026"
)

RuntimeError: ❌ GOOGLE_APPLICATION_CREDENTIALS is not set or file does not exist

In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

# Map the environment variable to the specific service_account section dlt expects
os.environ["DESTINATION__CREDENTIALS"] = os.getenv("GCP_CREDENTIALS", "")
os.environ["BUCKET_URL"] = os.getenv("BUCKET_URL", "gs://kestra-zoomcamp-de-demo-2026")

if not os.environ["DESTINATION__CREDENTIALS"]:
    print("❌ Error: Service Account credentials not found!")

In [ ]:
# # Install for production
# %%capture
# !pip install dlt[bigquery, gs]

In [ ]:
# # Install for testing
# %%capture
# !pip install dlt[duckdb]

In [12]:
import dlt
import requests
import pandas as pd
from dlt.destinations import filesystem
from io import BytesIO

Ingesting parquet files to GCS.

In [20]:
# Define a dlt source to download and process Parquet files as resources
@dlt.source(name="rides")
def download_parquet():
    prefix = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"
    for month in range(1, 7):
        file_name = f"yellow_tripdata_2024-0{month}.parquet"
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        # Return the dataframe as a dlt resource for ingestion
        yield dlt.resource(df, name=file_name)


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination=filesystem(layout="{schema_name}/{table_name}.{ext}"),
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
load_info = pipeline.run(download_parquet(), loader_file_format="parquet")

# Print the results
print(load_info)


PipelineStepFailed: Pipeline execution failed at `step=sync` with exception:

<class 'dlt.common.configuration.exceptions.InvalidNativeValue'>
`GcpOAuthCredentials` cannot parse the configuration value provided. The value is of type `str` and comes from the sections `('credentials',)` Value may be a secret and is not shown. Details:  The expected representation for `GcpOAuthCredentials` is a string with serialized oauth2 user info and may be wrapped in 'install'/'web' node - depending of oauth2 app type.

Ingesting data to Database

In [ ]:
# Define a dlt resource to download and process Parquet files as single table
@dlt.resource(name="rides", write_disposition="replace")
def download_parquet():
    prefix = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata'

    for month in range(1, 7):
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)

        df = pd.read_parquet(BytesIO(response.content))

        yield df


# Initialize the pipeline
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination="duckdb",  # Use DuckDB for testing
    # destination="bigquery",  # Use BigQuery for production
    dataset_name="rides_dataset",
)

# Run the pipeline to load Parquet data into DuckDB
info = pipeline.run(download_parquet)

# Print the results
print(info)


In [ ]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

# Set search path to the dataset
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

# Describe the dataset to see loaded tables
res = conn.sql("DESCRIBE").df()
print(res)

In [ ]:
# provide a resource name to query a table of that name
with pipeline.sql_client() as client:
    with client.execute_query(f"SELECT count(1) FROM rides") as cursor:
        data = cursor.df()
print(data)